<a href="https://colab.research.google.com/github/priyal6/AI-AGENT/blob/main/routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.1/500.1 kB 11.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.9
    Uninstalling langchain-core-1.2.9:
      Successfully uninstalled langchain-core-1.2.9


In [ ]:
from google.colab import userdata
import os
from langchain_openai import ChatOpenAI

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

In [ ]:
from typing import TypedDict
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.tools import tool
from langchain_openai import ChatOpenAI


class Context(TypedDict):
    user_role: str


@tool
def web_search(query: str) -> str:
    """Search the web for information."""
    return f"Fake search results for: {query}"

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""

    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt



model = ChatOpenAI(model="gpt-4.1")



agent = create_agent(
    model=model,
    tools=[web_search],
    middleware=[user_role_prompt],
    context_schema=Context,
)



result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Explain machine learning"}
        ]
    },
    context={"user_role": "expert"}
)

print(result["messages"][-1].content)

Machine learning is a branch of artificial intelligence (AI) that focuses on creating systems that can learn from data and improve their performance over time without being explicitly programmed for specific tasks.

Here’s a more detailed breakdown:

### Key Concepts

1. **Data**: Machine learning relies on large sets of data (examples, records, images, etc.) to "teach" an algorithm how to make decisions or predictions.

2. **Algorithms**: These are mathematical models or procedures that find patterns or relationships within the data.

3. **Training**: The process in which an algorithm processes data ("training set") and learns from it. The goal during training is for the model to find useful patterns (like distinguishing between spam and non-spam emails).

4. **Model**: After training, the algorithm becomes a "model" that can make predictions or decisions (e.g., classifying new emails as spam or not).

5. **Testing/Validation**: Separate data ("test set") are used to measure how well 

In [ ]:
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.tools import tool
from pydantic import BaseModel


class Context(TypedDict):
    user_role: str

class CalcInput(BaseModel):
    expression: str


@tool(args_schema=CalcInput)
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))


@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    user_role = request.runtime.context.get("user_role", "user")
    base = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base} Use precise mathematical reasoning."
    if user_role == "beginner":
        return f"{base} Explain calculations step by step."

    return base


model = ChatOpenAI(model="gpt-4.1")


agent = create_agent(
    model=model,
    tools=[calculator],
    middleware=[user_role_prompt],
    context_schema=Context,

)


result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is 7 * 9?"}
        ]
    },
    context={"user_role": "beginner"}
)

print(result["messages"][-1].content)

7 multiplied by 9 equals 63.


In [ ]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel

class ToolCall(BaseModel):
    tool: str
    expression: str

llm = ChatOpenAI(model="gpt-4o-mini")

def planner_node(state):
    response = llm.with_structured_output(ToolCall).invoke(
        "Decide the tool and arguments for: 7 * 9"
    )
    return {"tool_call": response}

def router(state):
  if state["tool_call"].tool == "calculator":
     return "calculator"
  return END

def calculator_node(state):
    expr = state["tool_call"].expression
    return {
        "result": str(eval(expr))
    }

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from pydantic import BaseModel



class GraphState(TypedDict, total=False):
    input: str
    tool: str
    expression: str
    result: str


class ToolCall(BaseModel):
    tool: str
    expression: str


llm = ChatOpenAI(model="gpt-4o-mini")


def planner_node(state: GraphState):
    response = llm.with_structured_output(ToolCall).invoke(
        f"Decide the tool and arguments for: {state['input']}. "
        "Available tool: calculator."
    )

    return {
        "tool": response.tool,
        "expression": response.expression,
    }



def router(state: GraphState):
    if state["tool"] == "calculator":
        return "calculator"
    return END


def calculator_node(state: GraphState):
    return {
        "result": str(eval(state["expression"]))
    }



graph = StateGraph(GraphState)

graph.add_node("planner", planner_node)
graph.add_node("calculator", calculator_node)

graph.set_entry_point("planner")

graph.add_conditional_edges("planner", router)
graph.add_edge("calculator", END)

app = graph.compile(debug=True)


output = app.invoke({"input": "7 * 9"})
print(output)

[values] {'input': '7 * 9'}
[updates] {'planner': {'tool': 'calculator', 'expression': '7 * 9'}}
[values] {'input': '7 * 9', 'tool': 'calculator', 'expression': '7 * 9'}
[updates] {'calculator': {'result': '63'}}
[values] {'input': '7 * 9', 'tool': 'calculator', 'expression': '7 * 9', 'result': '63'}
{'input': '7 * 9', 'tool': 'calculator', 'expression': '7 * 9', 'result': '63'}
